# Prepare Result DB

In [1]:
import sqlite3

import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer

/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset('xaviergillard/brabant-xvii-ner')
dataset = pd.concat([d.to_pandas() for _,d in dataset.items()]).set_index(["id"])
dataset

,tokens,labels
id,,
70,"[remissio, charles, etc, scavoir, faisons, a, ...","[O, B-PERS, O, O, O, O, O, O, O, O, O, O, O, O..."
145,"[kaerle, doen, te, wetene, allen, ende, eenen,...","[B-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
20,"[phls, etc., doen, te, wetene, allen, jegenwoi...","[B-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
316,"[phls, etc., doen, te, wetene, allen, jegenwoi...","[B-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
256,"[philips, bijder, gracien, godts, coninck., va...","[B-PERS, O, O, O, O, O, B-LOC, O, B-LOC, O, B-..."
...,...,...
5,"[abolitie, joes, thomaessoone, kaerle, etc., a...","[O, B-PERS, I-PERS, I-PERS, O, O, O, O, O, O, ..."
115,"[kaerle, etc., doen, condt, allen, luden, nu, ...","[B-PERS, O, O, O, O, O, O, O, O, O, O, O, O, O..."
366,"[remisie, albert, ende, isabel, etc., doen, te...","[O, B-PERS, O, B-PERS, O, O, O, O, O, O, O, O,..."


In [3]:
import pandas as pd

from typing import NamedTuple

class Entities(NamedTuple):
    persons:   set[str]
    dates:     set[str]
    locations: set[str]

    @staticmethod
    # supprime les répétitions et overlap
    def __no_overlap(s):
        l = sorted(list(s), key=len, reverse=True)
        r = []
        for e in l:
            for o in r:
                if str(e)in str(o):
                    break
            else:
                r.append(str(e))
        return set(r)

    def __or__(self, other: "Entities") -> "Entities":
        persons   = Entities.__no_overlap(self.persons | other.persons)
        dates     = Entities.__no_overlap(self.dates | other.dates)
        locations = Entities.__no_overlap(self.locations | other.locations)
        return Entities(persons, dates, locations)

    @staticmethod
    def from_pandas(df: pd.DataFrame) -> "Entities":
        persons   = { e for e in df[df["label"] == "PERS"]["entity"] }
        dates     = { e for e in df[df["label"] == "DATE"]["entity"] }
        locations = { e for e in df[df["label"] == "LOC"]["entity"] }
        return Entities(persons, dates, locations)

    @staticmethod
    def from_words_labels(tokens: list[str], labels: list[str]) -> "Entities":
        persons   = []
        dates     = []
        locations = []
        #
        current = None
        ctype   = None
        for t,l in zip(tokens, labels):
            if l == 'O':
                if current:
                    match ctype:
                        case 'PERS': persons.append(current)
                        case 'DATE': dates.append(current)
                        case 'LOC': locations.append(current)
                current = None
                ctype   = None
            elif l.startswith('B-'):
                if current:
                    match ctype:
                        case 'PERS': persons.append(current)
                        case 'DATE': dates.append(current)
                        case 'LOC': locations.append(current)
                current = t
                ctype   = l[2:]
            elif l in [f"I-{ctype}", "IGN"]:
                current = f"{current} {t}"
        #
        persons   = Entities.__no_overlap(persons)
        dates     = Entities.__no_overlap(dates)
        locations = Entities.__no_overlap(locations)
        #
        return Entities(persons, dates, locations)

    @staticmethod
    def from_tokenids_labels(tokens: list[int], wordids: list[int], labels: list[str], words: list[str]) -> "Entities":
        persons   = []
        dates     = []
        locations = []
        #
        current = None
        prevwid = None
        ctype   = None
        for t,w,l in zip(tokens, wordids, labels):
            if l == 'O':
                if current:
                    match ctype:
                        case 'PERS': persons.append(current)
                        case 'DATE': dates.append(current)
                        case 'LOC': locations.append(current)
                current = None
                ctype   = None
                prevwid = None
            elif l.startswith('B-'):
                match ctype:
                    case 'PERS': persons.append(current)
                    case 'DATE': dates.append(current)
                    case 'LOC': locations.append(current)
                current = words[w]
                ctype   = l[2:]
                prevwid = w
            elif l in [f"I-{ctype}", "IGN"]:
                if prevwid != w:
                    current = f"{current} {words[w]}"
                    prevwid = w
        #
        persons   = Entities.__no_overlap(persons)
        dates     = Entities.__no_overlap(dates)
        locations = Entities.__no_overlap(locations)
        #
        return Entities(persons, dates, locations)
        

In [6]:
def data_for(dataset: pd.DataFrame, name: str) -> dict[int, Entities]:
    vftok = AutoTokenizer.from_pretrained(name)
    vfmod = AutoModelForTokenClassification.from_pretrained(name).eval().cuda()
    result= dict()
    for doc_id, row in tqdm(dataset.iterrows()):
        encoded = vftok(
            list(row['tokens']), 
            max_length=510,
            padding=True,
            truncation=True,
            stride=20,
            is_split_into_words=True,
            return_overflowing_tokens=True,
        )
        data = []
        for i in range(len(encoded['input_ids'])):
            out   = vfmod(
                input_ids=torch.tensor(encoded['input_ids'][i], dtype=torch.int, device="cuda").unsqueeze(0), 
                attention_mask=torch.tensor(encoded['attention_mask'][i], dtype=torch.int, device="cuda").unsqueeze(0),
            )
            preds = out.logits.argmax(-1)[0]
            prds  = [ vfmod.config.id2label[l.item()] for l in preds ]
            ent   = Entities.from_tokenids_labels(encoded['input_ids'][i], encoded.encodings[i].word_ids, prds, row['tokens'])
            data.append(ent)
        # fold
        entities = Entities(set(), set(), set())
        for e in data:
            entities = entities | e
        result[doc_id] = entities
    return result

In [7]:
vieille_france    = data_for(dataset, "xaviergillard/xlm-roberta-large-vieille-france")
vieille_france_v2 = data_for(dataset, "xaviergillard/xlm-roberta-large-vieille-france-v2")
brabant_xvii_ner  = data_for(dataset, "xaviergillard/brabrant-xvii-ner")

419it [02:07,  3.28it/s]
419it [02:05,  3.35it/s]
419it [02:06,  3.31it/s]


In [11]:
import pickle

def save(o, fname):
    with open(fname, mode="wb") as f:
        pickle.dump(o, f)

save(vieille_france,    "./vieille_france.pickle")
save(vieille_france_v2, "./vieille_france_v2.pickle")
save(brabant_xvii_ner,  "./brabant_xvii_ner.pickle")

In [6]:
import pickle

def load(fname):
    with open(fname, mode="rb") as f:
        return pickle.load(f)

vieille_france    = load( "./vieille_france.pickle")
vieille_france_v2 = load( "./vieille_france_v2.pickle")
brabant_xvii_ner  = load( "./brabant_xvii_ner.pickle")

In [9]:
pd.DataFrame(vieille_france).T

,0,1,2
70,"{gilles de blaesvelt, hongrie de boheme, suppl...",{mars lan xve trentohuyct},"{brabant, bruxelles, reulx, pardeca, pardecha,..."
145,"{wylen gorys de scepere, willem nyet, j decock...",{},"{bruessel, loevenen, brabant}"
20,"{drancke, franchen claes, willem pauwels,, de ...",{iersten dach van septembri anno xvc achtentze...,"{bruessele, brabant, herentals, bezunderlyck, ..."
316,"{jan versluysen, bastiaen, de voirscreven, hen...",{},"{brusselle, strytbarich, twistich, brabant}"
256,"{passchier rogier, van beyde, philips bijder g...",{},"{berchem, brabant, portugael, grimbergen, hobo..."
...,...,...,...
5,"{joes thomaessoone, scamel huysman, aert van l...",{},{brabant}
115,"{anthonis lyff, anthonis weyggerde, anthonis w...","{xvc, xxxvii}","{antwerpen, sint joris, bruessele, brabant}"
366,"{adriaen van mechelen, christiaen van mechelen...",{},"{bruessela, herentals, brabant,}"
90,"{dair ghy, henrick van leve, henrick den voirs...",{},"{nederhespen, thienen, brabant, meyeren}"


In [10]:
conn = sqlite3.connect("unified.db")

In [14]:
cur = conn.execute("INSERT INTO models(name)  VALUES (?),(?);",
                  ('vieille_france_v2', 'brabant_xvii_ner'))

In [15]:
cur = conn.execute("select * from models;")
r = cur.fetchall()
print(r)

[('gpt4o-mini',), ('llama',), ('mistral7b',), ('mistralnemo',), ('vieille_france',), ('vieille_france_v2',), ('brabant_xvii_ner',)]


In [16]:
cur = conn.execute("delete from labeling where model = 'vieille_france';")

In [21]:
conn.close()

In [20]:
from json import dumps

def asdict(e):
    return {"dates": list(e.dates), "persons": list(e.persons), "locations": list(e.locations)}

def json(e):
    return dumps(asdict(e))

print(json(vieille_france_v2[16]))

{"dates": ["xve dach van augusto nu geleden vijff jaeren", "maent van meye int jaer onsheeren duysent vyffhondert vyffentzeventich"], "persons": ["kaerle", "jaspar smets", "mesdach", "voirs.verdonct", "hendrick verdonct", "vanden houte", "adriaens van ermighem", "philips"], "locations": ["bruessele", "spaengnien sicillien", "brabant limborch", "kelffs", "herent", "napels", "overmaze"]}


In [19]:
str(vieille_france_v2[16])

"Entities(persons={'kaerle', 'jaspar smets', 'mesdach', 'voirs.verdonct', 'hendrick verdonct', 'vanden houte', 'adriaens van ermighem', 'philips'}, dates={'xve dach van augusto nu geleden vijff jaeren', 'maent van meye int jaer onsheeren duysent vyffhondert vyffentzeventich'}, locations={'bruessele', 'spaengnien sicillien', 'brabant limborch', 'kelffs', 'herent', 'napels', 'overmaze'})"

In [25]:
def save_to_db(model, data):
    with sqlite3.connect("./unified.db") as conn:
        for k,v in data.items():
            _ = conn.execute("INSERT INTO labeling(id, model, labeling) VALUES (?, ?, ?)",
                                (k, model, json(v)))

save_to_db('vieille_france',    vieille_france)
save_to_db('vieille_france_v2', vieille_france_v2)
save_to_db('brabant_xvii_ner',  brabant_xvii_ner)

In [26]:
with sqlite3.connect('unified.db') as conn:
    cur = conn.execute("select count(distinct(id)) from labeling where model = 'vieille_france';")
    print(cur.fetchone())
    cur = conn.execute("select count(distinct(id)) from labeling where model = 'vieille_france_v2';")
    print(cur.fetchone())
    cur = conn.execute("select count(distinct(id)) from labeling where model = 'brabant_xvii_ner';")
    print(cur.fetchone())

(419,)
(419,)
(419,)
